## **BERTopic Untuk Topic Modelling**

In [ ]:
!pip install Sastrawi bertopic sentence-transformers transformers -q

In [ ]:
#1 import

import pandas as pd
import re
from bertopic import BERTopic
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import torch

## **Data**

In [ ]:
#2 konversi xlsx -> csv

excel_path = "1DATA_FIX.xlsx"
csv_path = "dataset.csv"

# Baca file Excel
df = pd.read_excel(excel_path, usecols=["Judul", "Abstrak"])

# Ambil kolom 'judul' dan 'abstrak'
df["text"] = df["Judul"].fillna('') + " " + df["Abstrak"].fillna('')

# Simpan jadi CSV
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"File berhasil dikonversi ke: {csv_path}")

File berhasil dikonversi ke: dataset.csv


In [ ]:
#3 load data

df = pd.read_csv(csv_path, usecols=["Judul", "Abstrak"])
print("Jumlah data:", len(df))
df.head()

Jumlah data: 235


,Judul,Abstrak
0,Perbandingan Metode Ceramah dan Metode Tutor ...,Abstrak Penelitian ini bertujuan untuk menguji...
1,Analisis Kemampuan Berpikir Kritis Siswa Melal...,Penelitian ini bertujuan untuk mengetahui (1) ...
2,Efektivitas Model Pembelajaran Guided Inquiry ...,Penelitian ini bertujuan untuk mengetahui (1) ...
3,Implementasi Model Pembelajaran Team Assisted ...,Penelitian ini bertujuan untuk mengetahui perb...
4,Pengaruh Penggunaan Media Pembelajaran Modul B...,Penelitian ini bertujuan untuk membuktikan ada...


## **Preprocessing**

In [ ]:
#4 preprocessing

# 1) Normalisasi nama kolom biar aman (hapus spasi, lowercase)
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'\s+', ' ', regex=True)
)

# 2) Deteksi kolom judul & abstrak (support variasi nama)
def find_col(candidates):
    for cand in candidates:
        for col in df.columns:
            if cand in col:  # match partial, mis. 'judul', 'title'
                return col
    return None

judul_col   = find_col(['judul', 'title'])
abstrak_col = find_col(['abstrak', 'abstract'])

if judul_col is None or abstrak_col is None:
    raise ValueError(
        f"Tidak menemukan kolom 'judul'/'abstrak'. Kolom tersedia: {list(df.columns)}"
    )

# 3) Buat kolom 'text' dari judul + abstrak
df['text'] = (
    df[judul_col].astype(str).str.strip() + '. ' +
    df[abstrak_col].astype(str).str.strip()
).str.replace(r'\s+', ' ', regex=True)

# Buang baris yang kosong setelah penggabungan
df = df[df['text'].str.strip().ne('.') & df['text'].str.strip().ne('')].copy()

# Cek
print(df[['text']].head())

                                                text
0  Perbandingan Metode Ceramah dan Metode Tutor S...
1  Analisis Kemampuan Berpikir Kritis Siswa Melal...
2  Efektivitas Model Pembelajaran Guided Inquiry ...
3  Implementasi Model Pembelajaran Team Assisted ...
4  Pengaruh Penggunaan Media Pembelajaran Modul B...


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["text"] = (df["judul"].fillna('') + " " + df["abstrak"].fillna(''))
df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
def split_abstract(text):
    sentences = text.split(".")
    return [s.strip() for s in sentences if len(s.split()) > 5]

docs = []
for abs in df["abstrak"].dropna():
    docs.extend(split_abstract(abs))

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "indobenchmark/indobert-base-p2"
    #"firqaaa/indo-sentence-bert-base"
)

embeddings = embedding_model.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

## **Topic Modelling**

In [ ]:
!pip install gensim -q

In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# ==================== STOPWORDS ====================
factory = StopWordRemoverFactory()
stopwords_id = factory.get_stop_words()

domain_stopwords = [
    "penelitian", "hasil", "metode", "data", "analisis",
    "menggunakan", "berdasarkan", "dilakukan", "dapat",
    "pengaruh", "signifikan", "positif", "negatif",
    "mahasiswa", "siswa", "peserta", "didik",
    "pembelajaran", "kelas", "universitas", "sebelas", "maret",
    "studi", "tujuan", "manfaat", "proses", "sistem",
    "aplikasi", "website", "media", "evaluasi", "efektivitas"
]

additional_stopwords = [
    "sangat", "terdapat", "memfasilitasi", "sebagai", "mempunyai",
    "berupa", "selain", "antar", "antara", "pihak", "para",
    "uns", "surakarta", "jawa", "tengah", "skripsi", "tugas", "akhir",
    "membangun", "merancang", "penerapan", "implementasi",
    "dikarenakan", "sehingga", "kemudian", "dengan", "untuk",
    "dari", "ini", "itu", "tersebut", "adalah", "yaitu", "eve", "ng", "er", "me", "strong"
]

final_stopwords = list(set(stopwords_id + domain_stopwords + additional_stopwords))

In [ ]:
import pandas as pd
import numpy as np
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from bertopic import BERTopic
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer

# Sekalian panggil stopwords
vectorizer_model = CountVectorizer(stop_words=final_stopwords, min_df=1, ngram_range=(1, 2))

# 1. Tokenisasi teks
analyzer = vectorizer_model.build_analyzer()
docs = df["clean_text"].tolist()
texts_tokenized = [analyzer(doc) for doc in docs]

# 2. Membuat Dictionary Gensim
gensim_dictionary = Dictionary(texts_tokenized)

# 3. Fungsi
def get_topic_words(topic_model, top_n=10):
    topic_words = []
    topics = topic_model.get_topic_info()
    valid_topics = topics[topics['Topic'] != -1]['Topic'].tolist()

    for topic_id in valid_topics:
        words = [word for word, _ in topic_model.get_topic(topic_id)[:top_n]]
        topic_words.append(words)

    return topic_words

# 4. Fungsi untuk menghitung Topic Diversity
def calculate_topic_diversity(topic_words):
    if not topic_words:
        return 0.0

    unique_words = set(word for topic in topic_words for word in topic)
    total_words = sum(len(topic) for topic in topic_words)

    diversity = len(unique_words) / total_words if total_words > 0 else 0
    return diversity

In [ ]:
import pandas as pd
import time
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from umap import UMAP
from bertopic import BERTopic
from hdbscan import HDBSCAN
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

# 1. Setup Data & Dictionary
docs = df["clean_text"].tolist()
texts = [doc.split() for doc in docs]
gensim_dictionary = Dictionary(texts)

# 2. Parameter Dasar Model
hdbscan_model = HDBSCAN(
    min_cluster_size=3,
    min_samples=1,
    metric="euclidean",
    cluster_selection_epsilon=0.05,
    prediction_data=True
)

representation_model = {
    "KeyBERT": KeyBERTInspired(),
    "MMR": MaximalMarginalRelevance(diversity=0.7)
}

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True, bm25_weighting=True)

# 3. Stopwords Setup
final_stopwords = list(set(stopwords_id + domain_stopwords + additional_stopwords))

seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
target_topics = 10
evaluation_results = []

# Mulai hitung total waktu
start_total = time.time()

for run_idx, seed in enumerate(seeds, start=1):
    run_start = time.time()
    print(f"\n{'='*40}")
    print(f"Memulai Run {run_idx}/10 dengan UMAP Seed: {seed}")

    vectorizer_model = CountVectorizer(stop_words=final_stopwords, ngram_range=(1, 2), min_df=1, max_df=0.9)
    umap_model_seeded = UMAP(n_neighbors=5, n_components=15, min_dist=0.01, metric="cosine", random_state=seed)

    # Inisialisasi
    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        umap_model=umap_model_seeded,
        hdbscan_model=hdbscan_model,
        representation_model=representation_model,
        ctfidf_model=ctfidf_model,
        top_n_words=10,
        calculate_probabilities=False,
        verbose=False
    )

    # Tahap 1: Fit Model
    topics, probs = topic_model.fit_transform(docs, embeddings)

    # Tahap 2: Reduksi Topik
    current_topics = len(set(topics)) - (1 if -1 in set(topics) else 0)
    if current_topics > target_topics:
        topic_model.reduce_topics(docs, nr_topics=target_topics)
        topics = topic_model.topics_

    # Tahap 3: Reduksi Outlier
    if -1 in topics:
        new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")
        topic_model.update_topics(
            docs,
            topics=new_topics,
            vectorizer_model=vectorizer_model,
            representation_model=representation_model
        )
        topics = new_topics

    # ==================== EVALUASI ====================
    topic_info = topic_model.get_topic_info()
    valid_topics = topic_info[(topic_info.Topic != -1) & (topic_info.Count >= 20)]["Topic"].tolist()

    if len(valid_topics) == 0:
        print(f"Run {run_idx} gagal: Tidak ada topik valid.")
        continue

    topics_words_ngrams = [[word for word, _ in topic_model.get_topic(tid)[:10]] for tid in valid_topics]
    topics_for_coherence = [[u for n in t for u in n.split()] for t in topics_words_ngrams]

    cv_model = CoherenceModel(topics=topics_for_coherence, texts=texts, dictionary=gensim_dictionary, coherence='c_v')
    c_v_score = cv_model.get_coherence()

    npmi_model = CoherenceModel(topics=topics_for_coherence, texts=texts, dictionary=gensim_dictionary, coherence='c_npmi')
    c_npmi_score = npmi_model.get_coherence()

    unique_words = set(word for topic in topics_words_ngrams for word in topic)
    total_words = sum(len(topic) for topic in topics_words_ngrams)
    diversity_score = len(unique_words) / total_words if total_words > 0 else 0

    quality_score = diversity_score * c_npmi_score

    run_end = time.time()
    run_duration = run_end - run_start
    print(f"Hasil Run {run_idx} - Valid Topics: {len(valid_topics)} | C_v: {c_v_score:.4f} | NPMI: {c_npmi_score:.4f} | Waktu: {run_duration:.2f} detik")

    evaluation_results.append({
        "Run": run_idx,
        "Seed": seed,
        "Valid_Topics_Count": len(valid_topics),
        "C_v": c_v_score,
        "C_NPMI": c_npmi_score,
        "Topic_Diversity": diversity_score,
        "Topic_Quality": quality_score,
        "Computation_Time": run_duration
    })

end_total = time.time()
print(f"\n{'='*40}")
print(f"Total Waktu Komputasi: {end_total - start_total:.2f} detik")

df_raw_results = pd.DataFrame(evaluation_results)
df_raw_results.to_csv("bertopic_10_runs_final.csv", index=False)
display(df_raw_results)


Memulai Run 1/10 dengan UMAP Seed: 10


2026-06-22 13:58:34,076 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Hasil Run 1 - Valid Topics: 3 | C_v: 0.5736 | NPMI: 0.0352 | Waktu: 37.21 detik

Memulai Run 2/10 dengan UMAP Seed: 20


2026-06-22 13:58:57,037 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Hasil Run 2 - Valid Topics: 3 | C_v: 0.4751 | NPMI: -0.0351 | Waktu: 20.15 detik

Memulai Run 3/10 dengan UMAP Seed: 30


...

2026-06-22 14:00:42,195 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Hasil Run 9 - Valid Topics: 4 | C_v: 0.5593 | NPMI: 0.0104 | Waktu: 14.31 detik

Memulai Run 10/10 dengan UMAP Seed: 100


2026-06-22 14:00:56,432 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Hasil Run 10 - Valid Topics: 3 | C_v: 0.5255 | NPMI: 0.0109 | Waktu: 14.13 detik

Total Waktu Komputasi: 175.98 detik


,Run,Seed,Valid_Topics_Count,C_v,C_NPMI,Topic_Diversity,Topic_Quality,Computation_Time
0,1,10,3,0.573582,0.035162,0.933333,0.032818,37.214224
1,2,20,3,0.475054,-0.035104,0.933333,-0.032764,20.153002
2,3,30,3,0.576390,0.031626,0.900000,0.028463,14.542204
3,4,40,2,0.556086,0.018253,0.950000,0.017341,13.820546
4,5,50,2,0.558819,0.024911,0.950000,0.023666,13.075720
5,6,60,4,0.468805,-0.069532,0.825000,-0.057364,16.651552
6,7,70,3,0.480582,-0.025720,0.933333,-0.024006,17.121348
7,8,80,4,0.481635,-0.023492,0.900000,-0.021142,14.947482
8,9,90,4,0.559312,0.010440,0.900000,0.009396,14.311444
9,10,100,3,0.525459,0.010850,0.900000,0.009765,14.125959


In [ ]:
# 1. Ekstrak kolom metrik yang tersedia di df_raw_results
metrics = ["C_v", "C_NPMI", "Topic_Diversity", "Topic_Quality", "Computation_Time"]

summary_data = []

for metric in metrics:
    if metric in df_raw_results.columns:
        data_metric = df_raw_results[metric]

        mean_val = data_metric.mean()
        std_val = data_metric.std()
        min_val = data_metric.min()
        max_val = data_metric.max()

        # Batas Error Bars
        lower_bound_sd = mean_val - std_val
        upper_bound_sd = mean_val + std_val

        summary_data.append({
            "Metric": metric,
            "Mean (Center)": mean_val,
            "Minimum": min_val,
            "Maximum": max_val,
            "Mean - SD (Lower)": lower_bound_sd,
            "Mean + SD (Upper)": upper_bound_sd,
            "Std_Dev": std_val
        })

# 2. Konversi ke DataFrame dan Ekspor ke CSV
df_summary = pd.DataFrame(summary_data)
df_summary.to_csv("bertopic_evaluation_summary.csv", index=False)

print("\n[INFO] Tabel Ringkasan Statistik berhasil disimpan sebagai 'bertopic_evaluation_summary.csv'")
display(df_summary)


[INFO] Tabel Ringkasan Statistik berhasil disimpan sebagai 'bertopic_evaluation_summary.csv'


,Metric,Mean (Center),Minimum,Maximum,Mean - SD (Lower),Mean + SD (Upper),Std_Dev
0,C_v,0.525573,0.468805,0.576390,0.481111,0.570034,0.044461
1,C_NPMI,-0.002260,-0.069532,0.035162,-0.036654,0.032133,0.034394
2,Topic_Diversity,0.912500,0.825000,0.950000,0.875388,0.949612,0.037112
3,Topic_Quality,-0.001383,-0.057364,0.032818,-0.031750,0.028985,0.030368
4,Computation_Time,17.596348,13.075720,37.214224,10.397683,24.795014,7.198665


In [ ]:
#lihat daftar topik
topic_model.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,0,115,0_pengembangan_berbasis_materi_ahli,"[pengembangan, berbasis, materi, ahli, belajar...","[model pengembangan, pendekatan, pengujian, ke...","[mengetahui, game, layak, wawancara, baik, ste...",[pengembangan game edukasi berbasis role playi...
1,1,65,1_belajar_learning_model_eksperimen,"[belajar, learning, model, eksperimen, uji, ni...","[hipotesis, learning tipe, based learning, ber...","[belajar, blended, mata pelajaran, mata, tekni...",[pengaruh inquiry based learning ibl berbantua...
2,2,26,2_algoritma_pengujian_akurasi_model,"[algoritma, pengujian, akurasi, model, lebih, ...","[pengujian, algoritma, cloud computing, prepro...","[algoritma, rata, resnet, waktu, mengetahui, k...",[sistem pembelajaran adaptif berbasis api untu...
3,3,10,3_pemrograman_mobile_python_pemrograman python,"[pemrograman, mobile, python, pemrograman pyth...","[postgresql mysql, pemrograman python, databas...","[pemrograman python, mobile pemrograman, kueri...",[pengaruh internet terhadap minat belajar sisw...
4,4,6,4_jaringan_virtual_proxmox_mikrotik,"[jaringan, virtual, proxmox, mikrotik, emulato...","[pengujian reliability, linearitas proxmox, tr...","[proxmox, mikrotik routerboard, mesin virtual,...",[uji perbandingan performa virtualisasi mesin ...
5,5,4,5_pekerjaan rumah_rumah_pekerjaan_pesawat,"[pekerjaan rumah, rumah, pekerjaan, pesawat, p...","[efektifitas pemberian, mindstorms ev, confide...","[rumah, pesawat sederhana, embel, pemberian pe...",[pengaruh penggunaan robotika pendidikan dalam...
6,6,3,6_game_permainan_activity_on,"[game, permainan, activity, on, game design, g...","[pengembangan permainan, konsep dasar, kemampu...","[game play, berpikir komputasi, minds on, desi...",[pengembangan media pembelajaran berbasis perm...
7,7,3,7_game_visual novel_novel_game visual,"[game, visual novel, novel, game visual, visua...","[menerangkan materi, tinjauan teori, memanfaat...","[visual novel, perang padri, roblox, matematik...",[analisis pemanfaatan media pembelajaraan mela...
8,8,3,8_means_publikasi_algoritma_algoritma means,"[means, publikasi, algoritma, algoritma means,...","[algoritma means, bidang kompetensi, algoritma...","[algoritma means, google scholar, bidang kompe...",[implementasi kombinasi algoritma k means dan ...


In [ ]:
#7 lihat kata kunci topik tertentu

topic_model.get_topic(0)

[('pengembangan', np.float64(0.022787007489316596)),
 ('berbasis', np.float64(0.017161575400621628)),
 ('materi', np.float64(0.0171512238585304)),
 ('ahli', np.float64(0.015665993046849503)),
 ('belajar', np.float64(0.014298299008345445)),
 ('teknik', np.float64(0.01424187827507848)),
 ('mengetahui', np.float64(0.013638213428011064)),
 ('kelayakan', np.float64(0.01329529094026399)),
 ('aspek', np.float64(0.01321607926000368)),
 ('game', np.float64(0.012776258544934986))]

## **Visualisasi**

In [ ]:
#visualisasi
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

In [ ]:
topic_model.visualize_documents(df['clean_text'].tolist(), topics=topics, embeddings=embeddings)

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
#9 simpan

df['topic'] = topics
df.to_csv("hasil_bertopic.csv", index=False, encoding="utf-8")
print("Hasil berhasil disimpan!")

Hasil berhasil disimpan!


## **CEK**

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info = topic_info[topic_info.Topic != -1]

In [ ]:
for _, row in topic_info.iterrows():
    topic_id = row["Topic"]
    freq = row["Count"]
    keywords = [word for word, _ in topic_model.get_topic(topic_id)[:5]]

    kalimat = (
        f"Topik {topic_id} merupakan topik yang sering muncul "
        f"dengan jumlah {freq} dokumen, "
        f"yang didominasi oleh kata kunci: {', '.join(keywords)}."
    )
    print(kalimat)

Topik 0 merupakan topik yang sering muncul dengan jumlah 115 dokumen, yang didominasi oleh kata kunci: pengembangan, berbasis, materi, ahli, belajar.
Topik 1 merupakan topik yang sering muncul dengan jumlah 65 dokumen, yang didominasi oleh kata kunci: belajar, learning, model, eksperimen, uji.
Topik 2 merupakan topik yang sering muncul dengan jumlah 26 dokumen, yang didominasi oleh kata kunci: algoritma, pengujian, akurasi, model, lebih.
Topik 3 merupakan topik yang sering muncul dengan jumlah 10 dokumen, yang didominasi oleh kata kunci: pemrograman, mobile, python, pemrograman python, mobile pemrograman.
Topik 4 merupakan topik yang sering muncul dengan jumlah 6 dokumen, yang didominasi oleh kata kunci: jaringan, virtual, proxmox, mikrotik, emulator.
Topik 5 merupakan topik yang sering muncul dengan jumlah 4 dokumen, yang didominasi oleh kata kunci: pekerjaan rumah, rumah, pekerjaan, pesawat, pesawat sederhana.
Topik 6 merupakan topik yang sering muncul dengan jumlah 3 dokumen, yang d

In [ ]:
top_10_topics = topic_info.head(10)

top_10_topics[["Topic", "Count"]]

,Topic,Count
0,0,115
1,1,65
2,2,26
3,3,10
4,4,6
5,5,4
6,6,3
7,7,3
8,8,3


In [ ]:
for _, row in top_10_topics.iterrows():
    topic_id = row["Topic"]
    freq = row["Count"]
    keywords = [word for word, _ in topic_model.get_topic(topic_id)[:5]]

    print(
        f"Topik {topic_id} ({freq} dokumen): "
        f"{', '.join(keywords)}"
    )

Topik 0 (115 dokumen): pengembangan, berbasis, materi, ahli, belajar
Topik 1 (65 dokumen): belajar, learning, model, eksperimen, uji
Topik 2 (26 dokumen): algoritma, pengujian, akurasi, model, lebih
Topik 3 (10 dokumen): pemrograman, mobile, python, pemrograman python, mobile pemrograman
Topik 4 (6 dokumen): jaringan, virtual, proxmox, mikrotik, emulator
Topik 5 (4 dokumen): pekerjaan rumah, rumah, pekerjaan, pesawat, pesawat sederhana
Topik 6 (3 dokumen): game, permainan, activity, on, game design
Topik 7 (3 dokumen): game, visual novel, novel, game visual, visual
Topik 8 (3 dokumen): means, publikasi, algoritma, algoritma means, scholar


In [ ]:
# Simpan model
embedding_model.save("my_embedding_model")
topic_model.save("my_bertopic_model", serialization="safetensors", save_embedding_model=False)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-22 14:01:14,977 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


## **KLASIFIKASI HASIL KE TIGA BIDANG**

In [ ]:
# ==================== DICTIONARY KATEGORI ====================
kata_kunci_kategori = {
    "Rekayasa Perangkat Lunak": [
        "kode", "pemrograman", "algoritma", "perangkat lunak", "pengembangan", "api", "kerangka kerja",
        "debug", "pengujian", "agile", "software", "coding", "program", "developer", "framework",
        "library", "repository", "git", "github", "testing", "debugging", "maintenance", "requirement",
        "design pattern", "oop", "object oriented", "scrum", "devops", "ci/cd", "docker", "kubernetes",
        "backend", "frontend", "fullstack", "mobile", "android", "ios", "web development", "database",
        "sql", "nosql", "api rest", "microservice"
    ],
    "Multimedia": [
        "gambar", "video", "audio", "grafis", "animasi", "multimedia", "visual", "suara", "desain", "ui/ux",
        "foto", "ilustrasi", "rendering", "3d", "2d", "modeling", "texture", "komposisi", "tipografi",
        "warna", "pencahayaan", "audio editing", "sound design", "music production", "video editing",
        "motion graphic", "efek visual", "vfx", "augmented reality", "virtual reality", "ar/vr",
        "game design", "interactive media", "user interface", "user experience", "ui design", "ux design",
        "responsive design", "prototyping", "adobe", "photoshop", "illustrator", "premiere", "after effects",
        "blender", "maya", "3ds max"
    ],
    "Jaringan Komputer": [
        "jaringan", "protokol", "router", "ip", "tcp", "http", "dns", "paket", "keamanan", "firewall",
        "nirkabel", "bandwidth", "server", "client", "switch", "hub", "bridge", "gateway", "modem",
        "access point", "ethernet", "wifi", "lan", "wan", "man", "vlan", "vpn", "nat", "dhcp", "dns server",
        "firewall rules", "ids", "ips", "enkripsi", "ssl", "tls", "ssh", "ftp", "smtp", "pop3", "imap",
        "http/https", "rest api", "soap", "web socket", "load balancing", "cloud computing", "aws", "azure",
        "gcp", "virtualisasi", "container", "docker network", "kubernetes network", "sdn", "nfv", "iot",
        "internet of things", "embedded system", "cybersecurity", "penetration testing"
    ],
    "Pendidikan": [
        "belajar", "mengajar", "pendidikan", "pembelajaran", "kurikulum", "materi", "siswa", "guru",
        "dosen", "mahasiswa", "sekolah", "universitas", "kuliah", "pelatihan", "workshop", "sertifikasi",
        "e-learning", "online learning", "moodle", "google classroom", "zoom", "teams", "presentasi",
        "tugas", "ujian", "nilai", "ijazah", "penelitian", "riset", "publikasi", "jurnal", "skripsi",
        "tesis", "disertasi", "bimbingan", "laboratorium", "praktikum", "modul", "buku ajar",
        "media pembelajaran", "teknologi pendidikan", "inovasi pembelajaran", "evaluasi", "asesmen", "learning", "stem", "konstruktivisme", "behaviorisme", "pekerjaan rumah"
    ]
}

# ==================== FUNGSI PEMETAAN ====================
def peta_topik_ke_kategori(kata_kunci_topik):
    """
    Memetakan daftar kata kunci suatu topik ke salah satu kategori.
    kata_kunci_topik: daftar tupel (kata, skor)
    """
    daftar_kata = [kata for kata, _ in kata_kunci_topik]
    teks_topik = " ".join(daftar_kata).lower()
    skor = {}
    for kategori, kunci in kata_kunci_kategori.items():
        skor[kategori] = sum(1 for kw in kunci if kw in teks_topik)
    terbaik = max(skor, key=skor.get)
    if skor[terbaik] > 0:
        return terbaik
    else:
        return "Lainnya"

# ==================== KLASIFIKASI DAN PENGUMPULAN HASIL ====================
hasil_klasifikasi = {}   # {id_topik: nama_bidang}
daftar_kata_per_topik = {} # {id_topik: list kata kunci (10 teratas)}

for id_topik in range(len(topic_model.get_topic_info())):
    if id_topik == -1:   # Lewati topik outlier
        continue
    kata_kunci = topic_model.get_topic(id_topik)
    if kata_kunci:
        bidang = peta_topik_ke_kategori(kata_kunci)
        hasil_klasifikasi[id_topik] = bidang
        daftar_kata_per_topik[id_topik] = [kata for kata, _ in kata_kunci[:10]]

# ==================== EKSPERIMEN LANJUTAN ====================
# 1. Simpulan topik per bidang
print("\n" + "="*60)
print("SIMPULAN TOPIK PER BIDANG")
print("="*60)

daftar_bidang = ["Rekayasa Perangkat Lunak", "Multimedia", "Jaringan Komputer", "Pendidikan", "Lainnya"]
for bidang in daftar_bidang:
    topik_dalam_bidang = [tid for tid, b in hasil_klasifikasi.items() if b == bidang]
    if not topik_dalam_bidang:
        continue
    print(f"\n=== BIDANG {bidang.upper()} ===")
    for tid in topik_dalam_bidang:
        kata_utama = daftar_kata_per_topik[tid][:5]   # 5 kata kunci teratas
        print(f"  Topik {tid}: {', '.join(kata_utama)}")

# 2. Hipotesis konteks
from collections import Counter
print("\n" + "="*60)
print("HIPOTESIS KONTEKS (KESIMPULAN UMUM)")
print("="*60)

hitungan_bidang = Counter(hasil_klasifikasi.values())
bidang_terbanyak = hitungan_bidang.most_common(1)[0][0]
print(f"- Bidang yang paling banyak muncul : {bidang_terbanyak} ({hitungan_bidang[bidang_terbanyak]} topik)")
print(f"- Total topik yang terklasifikasi  : {len(hasil_klasifikasi)}")
if "Lainnya" in hitungan_bidang:
    print(f"- Topik tidak terklasifikasi (Lainnya) : {hitungan_bidang['Lainnya']} topik")

# Persentase tiap bidang
print("\n--- Persentase tiap bidang ---")
for bidang in daftar_bidang:
    if bidang in hitungan_bidang:
        persen = (hitungan_bidang[bidang] / len(hasil_klasifikasi)) * 100
        print(f"  {bidang}: {persen:.1f}%")


SIMPULAN TOPIK PER BIDANG

=== BIDANG REKAYASA PERANGKAT LUNAK ===
  Topik 2: algoritma, pengujian, akurasi, model, lebih
  Topik 3: pemrograman, mobile, python, pemrograman python, mobile pemrograman
  Topik 8: means, publikasi, algoritma, algoritma means, scholar

=== BIDANG MULTIMEDIA ===
  Topik 6: game, permainan, activity, on, game design
  Topik 7: game, visual novel, novel, game visual, visual

=== BIDANG JARINGAN KOMPUTER ===
  Topik 4: jaringan, virtual, proxmox, mikrotik, emulator

=== BIDANG PENDIDIKAN ===
  Topik 0: pengembangan, berbasis, materi, ahli, belajar
  Topik 1: belajar, learning, model, eksperimen, uji
  Topik 5: pekerjaan rumah, rumah, pekerjaan, pesawat, pesawat sederhana

HIPOTESIS KONTEKS (KESIMPULAN UMUM)
- Bidang yang paling banyak muncul : Pendidikan (3 topik)
- Total topik yang terklasifikasi  : 9

--- Persentase tiap bidang ---
  Rekayasa Perangkat Lunak: 33.3%
  Multimedia: 22.2%
  Jaringan Komputer: 11.1%
  Pendidikan: 33.3%
